# Example: Run a grid of `MESA` models using `subprocesses`

In this example, we will see how to use the `morpy` package to prepare and run a grid of `MESA` models with `Python`. Specifically, we will see how to run the grid using the built-in `concurrent.futures` module. This will allow you to run your models in parallel on single-node machines. To run a grid using `MPI` (via `mpi4pi`), which allows you to run a grid over multiple nodes (i.e. on a cluster), please look at the `run_a_grid_with_mpi4pi` example.

Before reading this example, it is recommended to first go through the example called `prepare_and_run_single_model`.

As in the aforementioned example, we will look at the blocking (run) and non-blocking (launch) ways to compute `MESA` models in parallel. Since these grid computations can take a long time, the latter option is the more attractive one now. Therefore, we will also see how to track the status of our models, even when the `Python` session that launched them was terminated.

## 1. Imports

In [ ]:
from pathlib import Path # We use Path objects instead of strings for cleaner-looking paths to fiels and directories 
import morpy as morpy # The morpy package. We import the whole package for convenience, but separate impprts for the modules is also possible.
import numpy as np

## 2. General settings

The first few lines of the code block below are used to specify the `MESA` installation that you would like to use. These are similar to the environment variables you set in your `.*rc` files during the `MESA` and `MESASDK` installation (as specified here: https://docs.mesastar.org/en/26.4.1/installation.html) or in your module files. Even though the code has the capabilities to parse your `MESA` version from the `MESA` installation directory (here `MESA_DIR`), it does not hurt to specify it explicitly. Just make sure that the specified `MESA_VERSION` matches that of the `MESA` version installed at `MESA_DIR` to avoid conflicts.


The variable `PROJECT_DIR` is where your `MESA` project will be created. A `MESA` project consists of all the necessary files to compile the `Fortran` executables (`clean`, `mk`, `make/`, and `src/`), as well as the files to specify the history and profile output. Each `MESA` model gets its own directory within the project directory, where its inlists are stored and its output is written.


In [ ]:
# Adjust these paths to match your local desired MESA installation.
MESA_DIR = Path('<PATH_TO_MESA_DIR>')
MESASDK_ROOT = Path('<PATH_TO_MESASDK>')
OMP_THREADS  = 1 # Number of threads used per MESA model. Can be specified on a model-per-model basis when running/launching the models.
MESA_VERSION = 'r24.08.1'

## 2.1 Restart settings

Set `RESTART = True` if you are restarting a grid that was already partially prepared and/or run in a previous session (e.g. this notebook's kernel was restarted, or a previous `launch_grid`/`run_grid` call was interrupted). `RestartPolicy` decides, model by model, whether a rerun is needed based on its last known state.

`prepare_grid()` (used in section 4) checks `RESTART` against whether the tracking database at `project_dir/DB_NAME` already exists, and refuses to proceed if they disagree. This is the safety net against accidentally overwriting model directories/output from a forgotten `RESTART` flag — the database itself is never deleted automatically; to truly start fresh, delete it manually.


In [ ]:
# RESTART? -- set this to True if you want to resume earlier computations.
RESTART = False
DB_NAME = 'runs.db'

# Resume (restart) policy
restart_policy = morpy.RestartPolicy(
    enabled=True,                 # turn restart logic on/off
    match_policy="id_or_path",    # "id_only" or "id_or_path" (fallback match by model_path)
    skip_completed=True,          # don't rerun models already marked "completed"
    failed_policy="rerun",        # "rerun" or "skip" for models that ended "failed"
    cancelled_policy="rerun",     # "rerun" or "skip" for models that ended "cancelled"
    hydrate_timestamps=True,      # copy started_at/updated_at/finished_at from DB into Model
)


## 3. Set up `MESA` environment and parse default option files

In this step, which you only have to do once per project, we do two things. 

First, we set up the `MESA` environment, just as you would do in your `.*rc` or module file (e.g. `export MESA_DIR=...`), and create a `MesaConfig` object (called `mconf` in the code below). The latter holds information about the `MESA` installation, such as the `MESA` version in a normalised numeric format allowing the version-aware parts of the code to work properly. The `setup_mesa_env` function is a wrapper function calling two functions that can also be used separately (e.g. when you only want to create `MESA` inlists and do not need to compile it and run models, you only need the `MesaConfig` object): MesaConfig.setup() for the MesaConfig object and MesaConfig.set_mesa_env() for a dictionary containing the runtime environment for `MESA`.

Second, we parse the `MESA` default options files. These files usually live at `MESA_DIR/<star|binary|kap|eos>/defaults`. The resulting `DefaultsOptions` file (called `def_opts` here) contains `DefaultsFile` objects for all the different available default files in the `MESA` installation directory. These `DefaultsFile` objects can be explored, which comes in handy for creating inlists (see later).

Although many of the methods and functions used in the `morpy` package can generate the `MesaConfig` and `DefaultsOptions` objects on the fly, it is recommended to generate them once and pass them as an argument when a function or method requires it to avoid too much overhead.

In [ ]:
mconf = morpy.setup_mesa_env(mesa_dir=MESA_DIR, mesasdk_root=MESASDK_ROOT, omp_num_threads=OMP_THREADS, version_name=MESA_VERSION)
def_opts = morpy.build_defaults_options_from_config(mconf)
print('MESA version:', mconf.mesa_version.version_name)
print('     - year:', mconf.mesa_version.year)
print('     - month:', mconf.mesa_version.month)
print('     - patch:', mconf.mesa_version.patch)
print('     - scheme:', mconf.mesa_version.scheme)
print('     - revision:', mconf.mesa_version.revision)

## 4. Prepare a model grid and start tracking

In this example, we will run a small 2D grid of `MESA` models. Starting from a base model, that we import from an existing model directory, we will vary the initial mass and overshooting value.

We will put the each of the resulting `Model` objects into a `GridRunItem` object. In the next steps, these `GridRunItems` will be collected and passed onto the grid functions. Putting the `Model` objects into these `GridRunItem` objects might seem like a redundant layer of complexity, though the reason for the existence of the `GridRunItem` object class is simple. It has two attributes only: a `Model` and a `Mapping` attribute. The latter is the same kind of object as created by the `setup_mesa_env()` function used above and holds the environment variables required for running `MESA`. By including this as an extra attribute that is passed along to the grid functions, we can specify custom environment variables for each model separately. We can, for example if we know that some models in our grid are heavier to run than others, assign a higher number of threads (`OMP_NUM_THREADS`) to specific models. 

In [ ]:
# Import the basic work directory from the MESA installation directory
base_model = morpy.Model.empty(name='base')
# This function automatically imports all files inside the given directory containing the word "inlist" in their filename
morpy.import_inlists_from_dir(model=base_model, source_dir=MESA_DIR / 'star' / 'work', default_options=def_opts, config=mconf)

vars(base_model)

From the output of `vars(base_model)`, we see that the model currently has three inlists; `inlist`, `inlist_project`, and `inlist_pgstar`. Since we'll be running a grid, we are not interested in `PGPLOT` output, so we can remove this inlist from the model.

Afterwards, we'll make some changes to the remaining inlists before creating copies of this model.


In [ ]:
base_model.remove_inlist(inlist_name='inlist_pgstar')

inlist_header = base_model.inlists['inlist']
inlist_project = base_model.inlists['inlist_project']

inlist_header.set_option(name='read_extra_pgstar_inlist(1)', value=False, namelist='pgstar', defaults=def_opts)
inlist_project.set_option(name='pgstar_flag', value=False, namelist='star_job', defaults=def_opts)
inlist_project.remove_option(name='save_model_filename')
inlist_project.remove_option(name='stop_near_zams')
inlist_project.remove_option(name='Lnuc_div_L_zams_limit')
inlist_project.set_options(defaults=def_opts, controls={'xa_central_lower_limit(1)': '1d-6',
                                                        'overshoot_f0(1)': 0.004,
                                                        'overshoot_bdy_loc(1)': 'top',
                                                        'overshoot_scheme(1)': 'exponential',
                                                        'overshoot_zone_loc(1)': 'core',
                                                        'overshoot_zone_type(1)': 'burn_H'})



In [ ]:
# We can use this code block to preview the changes made to the inlists above
print('Contents of `inlist`:')
to_print = morpy.preview(base_model.inlists['inlist'].options, prefer_pandas=False)
print(to_print, '\n\n')
print('Contents of `inlist_project`:')
to_print = morpy.preview(base_model.inlists['inlist_project'].options, prefer_pandas=False)
print(to_print)


In [ ]:
initial_masses = np.arange(10, 18, 4)
overshoot_fs = np.arange(0.004, 0.016, 0.004)

name_template = 'm%s_fov%s'
model_dict = {}

for mi in initial_masses:
    for fov in overshoot_fs:
        name = name_template % (int(mi), str(fov).replace('.', 'p'))
        print(name)
        this_model = base_model.copy(model_copy_name=name, deep_copy=True)
        this_model.inlists['inlist_project'].set_options(defaults=def_opts, controls={'initial_mass': mi,
                                                                                      'overshoot_f(1)': fov})
        model_dict[name] = this_model

Although it would be more efficient to immediately wrap the resulting `Model` objects inside `GridRunItem` objects and create the necessary model directories, it is wiser to store the `Model` objects somewhere else (in this case in a dictionary) first. This allows us to inspect them to see if the requested changes to the inlists worked out as intended, before writing anything to our system storage. 

In [ ]:
print('Contents of `inlist_project`:')
to_print = morpy.preview(model_dict['m14_fov0p012'].inlists['inlist_project'].options, prefer_pandas=False)
print(to_print)

In [ ]:
# Create a project directory
project_dir = morpy.create_projectdir(Path('/Users/janhenneco/GitHub/MOrPy/docs/examples/example_output/run_a_grid/project_dir'), mode='star', config=mconf)

# Wrap each model in a GridRunItem
grid_list = [morpy.GridRunItem(model=model) for model in model_dict.values()]

# Create/validate model directories and the tracking DB together; raises if RESTART
# disagrees with whether project_dir/DB_NAME already exists.
# This function acts as your safeguard for accidnetally deleting your whole grid in case you forgot to
# put RESTART to True!
tracker = morpy.prepare_grid(project_dir, grid_list, restart=RESTART, db_name=DB_NAME, config=mconf)

print(f'{len(grid_list)} models prepared.')
for item in grid_list:
    print(f'  {item.model.name}  --->  {item.model.model_path}')

# Compile the executables
compile_result = morpy.compile_project(projectdir_path=project_dir, mode='star')
print(compile_result.pretty(2000))


## 5. Running and launching the grid

Now we'll see how to run (blocking) and launch (non-blocking) the grid we just set up. For this purpose, the `run_grid()` and `launch_grid()` function were created. Both functions take `backend` as an argument, which allows us to choose which parallelisation backend to use. The `local` backend uses the built-in `subprocesses` package to run multiple models in parallel and is great for single-node computations. The `mpi` backend uses `MPI` via `mpi4pi` to run or launch the grid. Using this backend requires the function to be called in a specific way, which we will learn about in the `run_a_grid_with_mpi` example.

As before, if we run the grid, we will have to wait until the grid has completely finished before we can continue working in our current Python session. If the Python session is closed, grid computations started by the `run_grid()` and `launch_grid()` functions will stop working, although some of the currently running models will likely continue running until they are finished. Nevertheless, this behaviour is undesirable. The main benefit of launching the grid vs. running it, is that we can launch the tracking functionalities set up above within the same Python session.  

In [ ]:
grid_handle = morpy.launch_grid(grid_list,
                            backend='local',
                            max_workers=3, # Number of models to run simultaneously
                            sync_before_run=True, # Sync any changes made to inlist files with Model objects in memory
                            default_options=def_opts,
                            config=mconf,
                            capture_output=True, # Store terminal output in individual models' RunResults
                            tracker=tracker,
                            restart_policy=restart_policy,
                            command_name='rn'
                            ) 

vars(grid_handle)

In [ ]:
# If you launched the grid (in a non-blocking way), 
# you can use following function to consult the status of the grid: 

morpy.print_grid_status(tracker=tracker, notebook=True)

In [ ]:
# If you launched the grid (in a non-blocking way), 
# you can use following function to terminate the grid:

stopped = morpy.stop_grid(handle=grid_handle, tracker=tracker)
print(stopped)

## 6. Tracking the status of the grid and terminating it

To make life easier, there are two CLI functions available to see the live status or stop the grid.
They need to be called from an active `morpy` environment and you need to know the location of the 
project directory and the name of the database file.

```
morpy_status <PATH_TO_PROJECT_DIR> <DATABASE_FILE_NAME> <REFRESH SECONDS>
```

```
morpy_stop <PATH_TO_PROJECT_DIR> <DATABASE_FILE_NAME>
```

You can also cancel a single model by PID (not so recommended; PIDs are not static) or by model ID (stable) in the following way:

```
morpy_cancel_model <PATH_TO_PROJECT_DIR> <DATABASE_FILE_NAME> --model-id <MODEL_ID>
```

or 

```
morpy_cancel_model <PATH_TO_PROJECT_DIR> <DATABASE_FILE_NAME> --pid <PID>
```